# 01 · Conditional GAN Training

Train a class-conditional GAN on MNIST. The Generator concatenates a learned class embedding with the latent noise vector, and the Discriminator fuses the same embedding with the image as an extra input channel. Loss is the standard BCE with logits — D pushes real logits toward 1 and fake logits toward 0, G uses the non-saturating form (label = 1 for its own samples).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import matplotlib.pyplot as plt

from data.dataloader import get_mnist_loaders
from model.cgan import Generator, Discriminator
from training.train_cgan import train_cgan
from utils.visualize import plot_image_grid, plot_training_history

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## Hyperparameters

In [ ]:
LATENT_DIM = 100
NUM_CLASSES = 10
BATCH_SIZE = 128
EPOCHS = 20            # bump to 30-50 for stronger samples
LR = 2e-4
BETAS = (0.5, 0.999)   # standard DCGAN choice

## Build the data + models

In [ ]:
train_loader, _ = get_mnist_loaders(batch_size=BATCH_SIZE, num_workers=2)

G = Generator(latent_dim=LATENT_DIM, num_classes=NUM_CLASSES).to(device)
D = Discriminator(num_classes=NUM_CLASSES).to(device)

n_params_g = sum(p.numel() for p in G.parameters())
n_params_d = sum(p.numel() for p in D.parameters())
print(f'G params: {n_params_g:,} | D params: {n_params_d:,}')

# Quick shape sanity check before training
z = torch.randn(4, LATENT_DIM, device=device)
y = torch.randint(0, NUM_CLASSES, (4,), device=device)
fake = G(z, y)
logits = D(fake, y)
print('fake:', fake.shape, '| D logits:', logits.shape)

## Train

Checkpoints are written under `checkpoints/cgan/` every 5 epochs and at the final epoch. Training prints the average D / G loss per epoch.

In [ ]:
history = train_cgan(
    G,
    D,
    train_loader,
    epochs=EPOCHS,
    lr=LR,
    beta1=BETAS[0],
    beta2=BETAS[1],
    latent_dim=LATENT_DIM,
    num_classes=NUM_CLASSES,
    device=device,
    checkpoint_dir='checkpoints/cgan',
    log_every=200,
)

In [ ]:
fig = plot_training_history(history, title='cGAN losses')
plt.show()

## Class-conditional samples

Generate one row per digit (0–9). If the Generator has learned the label conditioning, each row should look like the corresponding digit.

In [ ]:
G.eval()
n_per_class = 8
with torch.no_grad():
    z = torch.randn(NUM_CLASSES * n_per_class, LATENT_DIM, device=device)
    y = torch.arange(NUM_CLASSES, device=device).repeat_interleave(n_per_class)
    samples = G(z, y).cpu()

fig = plot_image_grid(samples, labels=y.cpu().tolist(), n_cols=n_per_class,
                     title='cGAN: 8 samples per digit class')
plt.show()

## Bonus: latent-space walk

Hold the class fixed and linearly interpolate between two noise vectors. A well-trained Generator should produce a smooth transition rather than jumping discretely between samples.

In [ ]:
def latent_walk(generator, label, steps=10, latent_dim=LATENT_DIM, device=device):
    z0 = torch.randn(1, latent_dim, device=device)
    z1 = torch.randn(1, latent_dim, device=device)
    alphas = torch.linspace(0, 1, steps, device=device).view(-1, 1)
    z = (1 - alphas) * z0 + alphas * z1
    y = torch.full((steps,), label, dtype=torch.long, device=device)
    with torch.no_grad():
        return generator(z, y).cpu()

walk = latent_walk(G, label=7, steps=10)
fig = plot_image_grid(walk, n_cols=10, title='Latent walk for class 7')
plt.show()